# Problem 04 — Personal Expense Categorizer

## Problem Statement
Digital payment platforms such as Paytm and PhonePe generate detailed transaction data, but users often lack tools to understand their spending patterns. The objective is to build a personal finance analytics tool that automatically categorizes transactions, identifies spending patterns, and presents useful insights through interactive visualizations.

## Required Approach
- **K-Means clustering** for behavioral grouping.
- **K-Nearest Neighbors (KNN)** for classifying new transactions.
- **Descriptive analytics** to understand spending patterns.
- **Plotly Express** for interactive visualizations.
- **Rule-based categorization** to create reliable transaction labels where appropriate.

## Deliverables
1. A functional categorization engine using real transaction data.
2. Interactive Plotly visualizations.
3. A documented GitHub-ready notebook/codebase.
4. A final presentation explaining the categorization logic and insights.

## Success Criterion
The analysis should reveal at least one useful, non-obvious spending insight that can encourage better financial decisions.

## 1. Import Required Libraries

Only libraries required for data preparation, machine learning, evaluation, and interactive visualization are used.

In [1]:
import numpy as np
import pandas as pd
import plotly.express as px

from scipy.sparse import csr_matrix, hstack
from sklearn.cluster import KMeans
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, silhouette_score
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

## 2. Load the Real Bank Transaction Dataset

The provided CSV is used as the real transaction source. The account identifier and cheque-number fields are not used as machine-learning features.

In [11]:
DATA_PATH = "bank-Sheet1.csv"

df = pd.read_csv(DATA_PATH)

# Standardize column names
df.columns = df.columns.str.strip()

# Remove columns that are identifiers or unnecessary for this project
df = df.drop(columns=["Account No", "CHQ.NO.", "."], errors="ignore")

# Clean transaction descriptions
df["TRANSACTION DETAILS"] = (
    df["TRANSACTION DETAILS"]
    .fillna("Unknown")
    .astype(str)
    .str.strip()
)

# Convert financial columns to numeric
for col in ["WITHDRAWAL AMT", "DEPOSIT AMT", "BALANCE AMT"]:
    df[col] = pd.to_numeric(
        df[col].astype(str).str.replace(",", "", regex=False),
        errors="coerce"
    )

# Convert dates
df["DATE"] = pd.to_datetime(df["DATE"], errors="coerce", dayfirst=True)
df["VALUE DATE"] = pd.to_datetime(df["VALUE DATE"], errors="coerce", dayfirst=True)

# Remove exact duplicate rows
df = df.drop_duplicates().reset_index(drop=True)

print(f"Dataset shape: {df.shape}")
print(f"Date range: {df['DATE'].min().date()} to {df['DATE'].max().date()}")

/tmp/ipykernel_629/529514828.py:27: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df["DATE"] = pd.to_datetime(df["DATE"], errors="coerce", dayfirst=True)
/tmp/ipykernel_629/529514828.py:28: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df["VALUE DATE"] = pd.to_datetime(df["VALUE DATE"], errors="coerce", dayfirst=True)


Dataset shape: (116162, 6)
Date range: 2015-01-01 to 2019-03-05


## 3. Create Transaction-Level Features

A withdrawal is treated as an outgoing transaction and a deposit as an incoming transaction. For spending analysis, only withdrawals are considered.

In [12]:
df["TRANSACTION TYPE"] = np.where(
    df["WITHDRAWAL AMT"].notna(),
    "Withdrawal",
    "Deposit"
)

df["TRANSACTION AMOUNT"] = (
    df["WITHDRAWAL AMT"].fillna(0)
    + df["DEPOSIT AMT"].fillna(0)
)

df["YEAR"] = df["DATE"].dt.year
df["MONTH"] = df["DATE"].dt.month
df["MONTH_NAME"] = df["DATE"].dt.month_name()
df["DAY_OF_WEEK"] = df["DATE"].dt.dayofweek
df["DAY_NAME"] = df["DATE"].dt.day_name()
df["IS_WEEKEND"] = df["DAY_OF_WEEK"].isin([5, 6]).astype(int)

expense_df = df[
    (df["TRANSACTION TYPE"] == "Withdrawal")
    & (df["TRANSACTION AMOUNT"] > 0)
].copy()

print(f"Total transactions: {len(df):,}")
print(f"Expense transactions: {len(expense_df):,}")
print(f"Deposit transactions: {(df['TRANSACTION TYPE'] == 'Deposit').sum():,}")

Total transactions: 116,162
Expense transactions: 53,525
Deposit transactions: 62,637


## 4. Rule-Based Transaction Categorization

Rules are applied to transaction descriptions to create meaningful spending categories. Bank transfers are separated from actual consumer spending so that moving money between accounts is not incorrectly counted as an expense.

In [13]:
def categorize_transaction(description):
    text = str(description).upper()

    # Financial transfers / movement of money
    if any(pattern in text for pattern in [
        "FDRL/INTERNAL FUND",
        "FDRL/NATIONAL ELECTRONIC",
        "FDRL/REAL TIME GROSS SETTL",
        "INTERNAL FUND TRANSFER",
        "TRF TO",
        "TRF FOR",
        "FUND TRF",
        "TRANSFER",
        "NEFT",
        "RTGS",
        "IMPS"
    ]):
        return "Bank Transfer"

    # Travel
    if "IRCTC" in text:
        return "Travel"

    # Entertainment
    if any(pattern in text for pattern in [
        "BIGTREE ENTERTAINMENT",
        "MEHAR ENTERTAINMENT",
        "ENTERTAINMENT"
    ]):
        return "Entertainment"

    # Shopping / retail
    if any(pattern in text for pattern in [
        "FUTURE RETAIL",
        "AXEL RETAIL",
        "RETAIL"
    ]):
        return "Shopping"

    # Utilities
    if any(pattern in text for pattern in [
        "BSES",
        "ELECTRICITY",
        "POWER"
    ]):
        return "Utilities"

    # Bank/service charges
    if any(pattern in text for pattern in [
        "SERVICE TAX",
        "CHARGE",
        "SCHRG",
        "LIEN MARKING",
        "FINANCE"
    ]):
        return "Financial Charges"

    # Cash withdrawals
    if any(pattern in text for pattern in [
        "ATM",
        "CASH"
    ]):
        return "Cash Withdrawal"

    # Payment gateways/services
    if any(pattern in text for pattern in [
        "PAYU",
        "INDIAIDEAS",
        "AVENUES"
    ]):
        return "Payment Services"

    return "Other"


expense_df["TRANSACTION CATEGORY"] = (
    expense_df["TRANSACTION DETAILS"]
    .apply(categorize_transaction)
)

# Actual spending dataset
spending_df = expense_df[
    expense_df["TRANSACTION CATEGORY"] != "Bank Transfer"
].copy()

print("Category distribution:")
print(spending_df["TRANSACTION CATEGORY"].value_counts())

Category distribution:
TRANSACTION CATEGORY
Other                20762
Payment Services      1918
Entertainment         1128
Travel                1080
Utilities              635
Financial Charges      533
Shopping               517
Cash Withdrawal         64
Name: count, dtype: int64


## 5. Descriptive Spending Analytics

These summaries answer the basic business question: **Where is the money going?**

In [14]:
category_summary = (
    spending_df
    .groupby("TRANSACTION CATEGORY")["TRANSACTION AMOUNT"]
    .agg(
        Transactions="count",
        Total_Spending="sum",
        Average_Transaction="mean",
        Median_Transaction="median",
        Maximum_Transaction="max"
    )
    .sort_values("Total_Spending", ascending=False)
)

category_summary["Spending_Percentage"] = (
    category_summary["Total_Spending"]
    / category_summary["Total_Spending"].sum()
    * 100
)

category_summary.round(2)

,Transactions,Total_Spending,Average_Transaction,Median_Transaction,Maximum_Transaction,Spending_Percentage
TRANSACTION CATEGORY,,,,,,
Other,20762,"17,085,586,238.82","822,925.84","2,124.00","459,447,546.36",96.25
Payment Services,1918,"336,908,544.25","175,656.18","43,520.50","20,151,893.57",1.90
Utilities,635,"132,610,979.38","208,836.19","5,020.00","13,435,536.00",0.75
Travel,1080,"122,478,908.71","113,406.40","54,048.00","2,099,625.00",0.69
Financial Charges,533,"45,303,678.58","84,997.52","18,000.00","9,710,101.00",0.26
Shopping,517,"12,812,976.00","24,783.32","3,075.00","1,780,340.00",0.07
Entertainment,1128,"7,708,383.97","6,833.67","3,334.50","265,034.00",0.04
Cash Withdrawal,64,"7,107,139.24","111,049.05","45,000.00","1,500,000.00",0.04


In [15]:
monthly_summary = (
    spending_df
    .groupby(["YEAR", "MONTH"])["TRANSACTION AMOUNT"]
    .agg(
        Transactions="count",
        Total_Spending="sum",
        Average_Transaction="mean"
    )
    .reset_index()
)

monthly_summary["PERIOD"] = (
    monthly_summary["YEAR"].astype(str)
    + "-"
    + monthly_summary["MONTH"].astype(str).str.zfill(2)
)

monthly_summary.sort_values(["YEAR", "MONTH"]).round(2)

,YEAR,MONTH,Transactions,Total_Spending,Average_Transaction,PERIOD
0,2015,1,14,"54,208.00","3,872.00",2015-01
1,2015,2,18,"29,742.00","1,652.33",2015-02
2,2015,3,11,"26,472.00","2,406.55",2015-03
3,2015,4,15,"54,428,736.00","3,628,582.40",2015-04
4,2015,5,20,"99,390,236.00","4,969,511.80",2015-05
5,2015,6,31,"483,969,669.13","15,611,924.81",2015-06
6,2015,7,134,"144,022,633.49","1,074,795.77",2015-07
7,2015,8,248,"431,768,131.74","1,741,000.53",2015-08
8,2015,9,249,"221,907,824.87","891,196.08",2015-09
9,2015,10,306,"274,524,252.62","897,138.08",2015-10


## 6. K-Means Clustering — Behavioral Spending Groups

K-Means groups transactions with similar numerical characteristics. The elbow curve is used to select a reasonable number of clusters.

In [16]:
cluster_features = spending_df[
    ["TRANSACTION AMOUNT", "MONTH", "DAY_OF_WEEK", "IS_WEEKEND"]
].copy()

cluster_scaler = StandardScaler()
X_cluster = cluster_scaler.fit_transform(cluster_features)

# Elbow method
k_values = range(2, 9)
inertias = []

for k in k_values:
    model = KMeans(n_clusters=k, random_state=42, n_init=10)
    model.fit(X_cluster)
    inertias.append(model.inertia_)

elbow_df = pd.DataFrame({
    "K": list(k_values),
    "Inertia": inertias
})

fig = px.line(
    elbow_df,
    x="K",
    y="Inertia",
    markers=True,
    title="K-Means Elbow Method"
)
fig.show()

In [17]:
# Select K=4 for a compact and interpretable behavioral segmentation
K = 4

kmeans = KMeans(
    n_clusters=K,
    random_state=42,
    n_init=10
)

spending_df["KMEANS_CLUSTER"] = kmeans.fit_predict(X_cluster)

cluster_summary = (
    spending_df
    .groupby("KMEANS_CLUSTER")["TRANSACTION AMOUNT"]
    .agg(
        Transactions="count",
        Average_Amount="mean",
        Median_Amount="median",
        Total_Spending="sum",
        Maximum_Amount="max"
    )
    .sort_values("Average_Amount")
)

cluster_summary.round(2)

,Transactions,Average_Amount,Median_Amount,Total_Spending,Maximum_Amount
KMEANS_CLUSTER,,,,,
0,12118,"524,732.06","3,000.00","6,358,703,058.77","58,500,000.00"
1,13108,"628,810.78","4,000.00","8,242,451,647.09","103,020,294.20"
3,1407,"1,119,095.28","2,300.00","1,574,567,055.18","150,000,000.00"
2,4,"393,698,771.98","424,103,611.57","1,574,795,087.91","459,447,546.36"


In [18]:
# Quantitative validation of the selected clustering
silhouette = silhouette_score(
    X_cluster,
    spending_df["KMEANS_CLUSTER"],
    sample_size=min(10000, len(spending_df)),
    random_state=42
)

print(f"Silhouette score for K={K}: {silhouette:.3f}")

fig = px.scatter(
    spending_df.sample(
        min(10000, len(spending_df)),
        random_state=42
    ),
    x="MONTH",
    y="TRANSACTION AMOUNT",
    color="KMEANS_CLUSTER",
    hover_data=[
        "TRANSACTION DETAILS",
        "TRANSACTION CATEGORY"
    ],
    title="K-Means Spending Behavior Groups"
)
fig.show()

Silhouette score for K=4: 0.356


## 7. KNN — Classify New Transactions

KNN is used to classify a new transaction into one of the existing spending categories.

- **TF-IDF** represents transaction descriptions as text features.
- Standardized numerical features add transaction amount and time information.
- The model is evaluated on unseen transactions.

In [19]:
knn_data = spending_df[
    [
        "TRANSACTION DETAILS",
        "TRANSACTION AMOUNT",
        "MONTH",
        "DAY_OF_WEEK",
        "TRANSACTION CATEGORY"
    ]
].dropna().copy()

# Keep the KNN dataset manageable while retaining a representative sample
MAX_KNN_ROWS = 40000

if len(knn_data) > MAX_KNN_ROWS:
    knn_data = (
        knn_data.groupby("TRANSACTION CATEGORY", group_keys=False)
        .apply(
            lambda group: group.sample(
                n=max(1, round(MAX_KNN_ROWS * len(group) / len(knn_data))),
                random_state=42
            )
        )
        .reset_index(drop=True)
    )

X_text = knn_data["TRANSACTION DETAILS"]
y = knn_data["TRANSACTION CATEGORY"]

X_train_text, X_test_text, y_train, y_test = train_test_split(
    X_text,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

text_vectorizer = TfidfVectorizer(
    lowercase=True,
    max_features=5000,
    min_df=2,
    ngram_range=(1, 2)
)

X_train_text = text_vectorizer.fit_transform(X_train_text)
X_test_text = text_vectorizer.transform(X_test_text)

numeric_columns = [
    "TRANSACTION AMOUNT",
    "MONTH",
    "DAY_OF_WEEK"
]

train_indices = y_train.index
test_indices = y_test.index

numeric_scaler = StandardScaler()

X_train_numeric = numeric_scaler.fit_transform(
    knn_data.loc[train_indices, numeric_columns]
)

X_test_numeric = numeric_scaler.transform(
    knn_data.loc[test_indices, numeric_columns]
)

X_train = hstack([
    X_train_text,
    csr_matrix(X_train_numeric)
]).tocsr()

X_test = hstack([
    X_test_text,
    csr_matrix(X_test_numeric)
]).tocsr()

knn = KNeighborsClassifier(
    n_neighbors=5,
    weights="distance",
    metric="cosine",
    n_jobs=-1
)

knn.fit(X_train, y_train)

y_pred = knn.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)

print(f"KNN Accuracy: {accuracy:.3f}")

KNN Accuracy: 0.997


In [20]:
print("Classification Report:\n")
print(classification_report(y_test, y_pred, zero_division=0))

labels = sorted(y_test.unique())
cm = confusion_matrix(y_test, y_pred, labels=labels)

fig = px.imshow(
    cm,
    x=labels,
    y=labels,
    text_auto=True,
    aspect="auto",
    labels={
        "x": "Predicted Category",
        "y": "Actual Category",
        "color": "Transactions"
    },
    title="KNN Confusion Matrix"
)
fig.show()

Classification Report:

                   precision    recall  f1-score   support

  Cash Withdrawal       1.00      0.54      0.70        13
    Entertainment       1.00      1.00      1.00       226
Financial Charges       1.00      0.94      0.97       106
            Other       1.00      1.00      1.00      4153
 Payment Services       1.00      1.00      1.00       384
         Shopping       1.00      1.00      1.00       103
           Travel       1.00      1.00      1.00       216
        Utilities       1.00      0.99      1.00       127

         accuracy                           1.00      5328
        macro avg       1.00      0.93      0.96      5328
     weighted avg       1.00      1.00      1.00      5328



## 8. Predict the Category of a New Transaction

In [21]:
def classify_new_transaction(description, amount, month, day_of_week):
    text_features = text_vectorizer.transform([str(description)])

    numeric_features = numeric_scaler.transform([
        [amount, month, day_of_week]
    ])

    features = hstack([
        text_features,
        csr_matrix(numeric_features)
    ]).tocsr()

    return knn.predict(features)[0]


# Example new transaction
example_category = classify_new_transaction(
    description="PAYU PAYMENTS PRIVATE LIMITED",
    amount=850,
    month=10,
    day_of_week=3
)

print("Predicted category:", example_category)

Predicted category: Payment Services


/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning:

X does not have valid feature names, but StandardScaler was fitted with feature names



## 9. Interactive Plotly Dashboard Views

These visualizations directly support the project objective of presenting spending insights to the user.

In [22]:
# Category spending
category_plot = (
    spending_df
    .groupby("TRANSACTION CATEGORY")["TRANSACTION AMOUNT"]
    .sum()
    .reset_index()
    .sort_values("TRANSACTION AMOUNT", ascending=False)
)

fig = px.bar(
    category_plot,
    x="TRANSACTION CATEGORY",
    y="TRANSACTION AMOUNT",
    title="Total Spending by Category",
    labels={
        "TRANSACTION CATEGORY": "Category",
        "TRANSACTION AMOUNT": "Spending (₹)"
    }
)
fig.show()


# Monthly spending
fig = px.line(
    monthly_summary,
    x="PERIOD",
    y="Total_Spending",
    markers=True,
    title="Monthly Spending Trend",
    labels={
        "PERIOD": "Month",
        "Total_Spending": "Spending (₹)"
    }
)
fig.show()


# Spending distribution
fig = px.pie(
    category_plot,
    names="TRANSACTION CATEGORY",
    values="TRANSACTION AMOUNT",
    title="Spending Distribution by Category"
)
fig.show()

## 10. Final Insights

The final presentation should report:

1. **Highest spending category** — the category with the largest total spending.
2. **Monthly trend** — months with unusually high or low spending.
3. **Behavioral groups** — what the K-Means clusters indicate about transaction behavior.
4. **KNN performance** — accuracy and category-level performance on unseen transactions.
5. **Actionable recommendation** — one concrete change the user could make based on the discovered pattern.

In [23]:
top_category = category_summary.index[0]
top_category_amount = category_summary.iloc[0]["Total_Spending"]
top_category_share = category_summary.iloc[0]["Spending_Percentage"]

highest_month = monthly_summary.loc[
    monthly_summary["Total_Spending"].idxmax()
]

print("FINAL PROJECT INSIGHTS")
print("=" * 50)
print(f"Highest spending category : {top_category}")
print(f"Category spending         : ₹{top_category_amount:,.2f}")
print(f"Share of total spending   : {top_category_share:.2f}%")
print(
    f"Highest spending month    : {highest_month['PERIOD']} "
    f"(₹{highest_month['Total_Spending']:,.2f})"
)
print(f"KNN test accuracy          : {accuracy:.2%}")
print(f"K-Means silhouette score   : {silhouette:.3f}")

FINAL PROJECT INSIGHTS
Highest spending category : Other
Category spending         : ₹17,085,586,238.82
Share of total spending   : 96.25%
Highest spending month    : 2018-01 (₹1,215,440,900.83)
KNN test accuracy          : 99.72%
K-Means silhouette score   : 0.356


## Conclusion

This notebook implements the complete machine-learning workflow required by **Problem 04 — Personal Expense Categorizer**:

**Real transaction data → preprocessing → rule-based categorization → descriptive analytics → K-Means behavioral grouping → KNN classification → Plotly interactive insights**

The resulting analysis can be connected to a Streamlit/FastAPI dashboard for the deployment deliverable.